# 01 — EDA and export

Export Discord messages from MySQL to local Parquet, profile the corpus, and build preceding-message context windows.

**Prerequisites:** `.env` with `MYSQL_*` credentials. See `.env.example`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Allow running without `pip install -e .`
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from sentiment.config import ensure_data_dirs, get_pipeline_config
from sentiment.db import export_messages_parquet, load_messages_parquet
from sentiment.estimate import estimate_from_windows_df
from sentiment.windows import export_windows_parquet, load_windows_parquet

pd.set_option("display.max_colwidth", 120)
paths = ensure_data_dirs()
cfg = get_pipeline_config(require_mysql=True)
print("Model:", cfg.model)
print("Context size:", cfg.context_size)
print("Data dir:", paths["data"])

## Export messages

In [ ]:
messages_path = export_messages_parquet(cfg.mysql)
messages = load_messages_parquet(messages_path)
print(f"Exported {len(messages):,} rows → {messages_path}")
messages.head()

## Corpus profile

In [ ]:
print("Total:", len(messages))
print("Empty:", int(messages["is_empty"].sum()))
print("Scorable:", int((~messages["is_empty"]).sum()))
print("Date range:", messages["created_at"].min(), "→", messages["created_at"].max())
print("Avg chars:", round(messages.loc[~messages["is_empty"], "content_len"].mean(), 1))
print("Channels:", messages["channel_name"].nunique())
print("Members:", messages["member_id"].nunique())

In [ ]:
channel_counts = messages["channel_name"].value_counts()
channel_counts.head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
messages.loc[~messages["is_empty"], "content_len"].clip(upper=300).hist(
    bins=40, ax=axes[0], color="#2a6f7f"
)
axes[0].set_title("Content length (clipped at 300)")
axes[0].set_xlabel("chars")

yearly = messages.assign(year=messages["created_at"].dt.year).groupby("year").size()
yearly.plot(kind="bar", ax=axes[1], color="#c45c26")
axes[1].set_title("Messages per year")
axes[1].set_xlabel("year")
plt.tight_layout()
plt.show()

## Build context windows

Each scorable message gets up to `CONTEXT_SIZE` preceding messages from the same channel.

In [ ]:
windows_path = export_windows_parquet(
    messages,
    context_size=cfg.context_size,
    max_content_chars=cfg.max_content_chars,
)
windows = load_windows_parquet(windows_path)
print(f"Windows: {len(windows):,} → {windows_path}")
windows[["message_id", "channel_name", "author_name", "n_priors", "context_text"]].head(3)

In [ ]:
# Example context block
print(windows.iloc[100]["context_text"] if len(windows) > 100 else windows.iloc[0]["context_text"])

## Full-corpus cost estimate (do not run yet)

In [ ]:
est = estimate_from_windows_df(windows, batch_size=cfg.batch_size)
est.as_dict()

Next: open `02_eval_sample.ipynb` and score a stratified 1k sample before committing to the full run.